# Experiments 73-83
**Prueba de hiperparámetros:** multi_scale / weight_decay / dropout / momentum

Se explorarán distintos mezclas de hiperparámetros para encontrar la mejor configuración para el dataset.

- **Model:**
    1. `yolov8m` *(Medium)*
- **Dataset:** 3.5m | 90º ***(v5i)***
- **Crop Tiles:** 640px
- **Sizes:** small & mid
- **Experiments:**
    - **Reference:** Roboflow 9x (72)

## Init

In [ ]:
import os
import shutil
import fnmatch
import pickle

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 85.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling 

## Helper Functions

In [ ]:
# Change for different file formats
reference = {
  "small": {
    "suffix": ".S",
    "file": "209"
  },
  "mid": {
    "suffix": ".M",
    "file": "503"
  },
  "large": {
    "suffix": ".L",
    "file": "000"
  }
}

In [ ]:
# Clone config files
def copy_config(src_folder, dest_folder):
    """
    Copies files from src_folder to dest_folder, excluding subfolders.

    Args:
        src_folder: The path to the source folder.
        dest_folder: The path to the destination folder.
    """

    try:
        # Ensure destination folder exists
        os.makedirs(dest_folder, exist_ok=True)

        for filename in os.listdir(src_folder):
            src_path = os.path.join(src_folder, filename)
            dest_path = os.path.join(dest_folder, filename)

            if os.path.isfile(src_path):
                shutil.copy2(src_path, dest_path) #copy metadata as well.
                #Use shutil.copy for not copying metadata.
                print(f"Copied: {filename}")
            #else: #optional
                #print(f"Skipped (not a file): {filename}") #optional. Uncomment if you want to see the skipped folders.

        print("✅ Copying complete.")

    except Exception as e:
        print(f"❌ An error occurred: {e}")


In [ ]:
# Copy filtered dataset images/labels
def copy_and_filter_folder(src_folder, dest_folder, pattern):
    """
    Copies a folder and files that match the given pattern.
    Alerts the user when a folder or file already exists but *does not* overwrite.
    Creates only what is needed.

    :param src_folder: Path to the source folder.
    :param dest_folder: Path to the destination folder.
    :param pattern: Filename pattern to keep (e.g., "*.txt").
    """
    try:
        # Ensure destination folder exists
        if not os.path.exists(dest_folder):
            print(f"✓ Creating destination folder '{dest_folder}'.\n")
            os.makedirs(dest_folder)
        else:
            print(f"✓ Destination folder '{dest_folder}' already exists.\n")

        # Walk through the source folder
        for root, _, files in os.walk(src_folder):
            relative_path = os.path.relpath(root, src_folder)
            new_root = os.path.join(dest_folder, relative_path)

            if not os.path.exists(new_root):
                print(f"Creating subdirectory '{new_root}'")
                os.makedirs(new_root)
            else:
                print(f"❕Subdirectory '{new_root}' already exists.")
                print("Make sure the data inside is relevant. Otherwise, just delete the folder and repeat the cloning process.")

            for file in files:
                if fnmatch.fnmatch(file, pattern + "*"):
                    src_file = os.path.join(root, file)
                    dest_file = os.path.join(new_root, file)

                    if not os.path.exists(dest_file):
                        shutil.copy2(src_file, dest_file)  # copy metadata as well
                    else:
                        print(f"❗️File '{dest_file}' already exists. Skipping.")

            print(f" ✓ Copying files complete.\n")
        print("✅ Copying dataset complete.")

    except Exception as e:
        print(f"❌ An error occurred: {e}")

In [ ]:
def copy_directory(source, destination, overwrite=True):
    try:
        if overwrite and os.path.exists(destination):
            shutil.rmtree(destination)  # Remove existing directory
        shutil.copytree(source, destination)
        print(f"Directory '{source}' copied to '{destination}'")
        return True  # Indicate success
    except FileExistsError:
        print(f"Destination '{destination}' exists. Use overwrite=True to replace.")
        return False  # Indicate failure

In [ ]:
# Load/Download prediction results (BB + confidence) as .pkl file

def save_results(results, filename):
    with open(filename, 'wb') as f:
        pickle.dump(results, f)

def load_results(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)

In [ ]:
def save_on_cloud(source: str, destination: str):
    """
    Saves a folder to a cloud storage location (e.g., Google Drive in Colab).

    Args:
        source (str): The path to the source folder.
        destination (str): The path to the destination folder (in cloud storage).
    """
    # 0. Input Validation (Assertions)
    assert isinstance(source, str), "Source must be a string."
    assert isinstance(destination, str), "Destination must be a string."

    try:
        # 1. Verify Source Folder
        if not os.path.exists(destination):
            os.makedirs(destination)

        # 2. Copy the Folder
        shutil.copytree(source, destination, dirs_exist_ok=True)
        print("✅ Folder copied successfully:\n  ",source,"\n  -->",destination)

    except Exception as e:
        print(f"❌ An error occurred: {e}")

### Validation functions

In [ ]:
cm = lambda results: results.confusion_matrix.matrix.tolist() if hasattr(results, 'confusion_matrix') and hasattr(results.confusion_matrix, 'matrix') else None

In [ ]:
import json

def save_json(results):
  try:
    data_to_store = {
        "confusion_matrix": cm(results),
        "results_dict": results.results_dict,
        "speed": results.speed
    }

    # Convert the Python dictionary to a JSON string
    json_data = json.dumps(data_to_store, indent=4)

    # You can now save this JSON string to a file
    folder = str(results.save_dir)
    with open(f"/content/{folder}/results.json", "w") as f:
        f.write(json_data)

    print("✅ JSON file stored in:", folder)

  except Exception as e:
      print(f"❌ An error occurred: {e}")

  #return json_data


In [ ]:
def gimme_metrics(results):
  matrix = cm(results)
  total_det = sum(sum(value) for value in matrix)
  percentages = []
  for row in matrix:
      values_percentages = []
      for value in row:
          if total_det != 0:
              percentage = (value / total_det) * 100
          else:
              percentage = 0.0
          values_percentages.append(f"{percentage:.2f}%")
      percentages.append(values_percentages)

  print("Total objects detected:", total_det)
  print("Confusion matrix:")
  for row in percentages:
      print(row)

  return matrix


In [ ]:
def show_cm(TP, FP, FN):
    matrix = [[TP, FP], [FN, 0]]
    total_det = sum(sum(value) for value in matrix)
    percentages = []
    for row in matrix:
        values_percentages = []
        for value in row:
            if total_det != 0:
                percentage = (value / total_det) * 100
            else:
                percentage = 0.0
            values_percentages.append(f"{percentage:.2f}%")
        percentages.append(values_percentages)

    print("Total objects detected:", total_det)
    print("\nConfusion matrix:")
    for row in percentages:
        a, b = row
        print(f"[ {a} , {b} ]")

In [ ]:
def show_metrics(TP, FP, FN):
    show_cm(TP, FP, FN)
    accuracy = TP/(TP+FP+FN)
    precision = TP/(TP+FP)
    recall = TP/(TP+FN)
    f1 = 2 * (precision * recall) / (precision + recall)
    f2 = 1.25 * (precision * recall) / (0.25 * precision + recall)
    fm = (precision * recall) ** 0.5
    print("\nMetrics:")
    print(f"- Accuracy: {accuracy:.3f}")
    print(f"- Precision: {precision:.3f}")
    print(f"- Recall: {recall:.3f}")
    print(f"- F1 Score: {f1:.3f}")
    print(f"- F½ Score: {f2:.3f}")
    print(f"- G-mean: {fm:.3f}")

# Datasets builder

## Importing from Drive

In [ ]:
!rm -rf /content/sample_data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Check if the cloud path is ok and the dataset can be found
!ls /content/drive/MyDrive/YOLO

3.5m.v3i.yolov8.640px		       3.5m.v4i.yolov8.640px.aug.v1
3.5m.v3i.yolov8.640px.aug.v1	       3.5m.v4i.yolov8_blended.640px
3.5m.v3i.yolov8.640px.aug.v1.soil_aug  3.5m.v4i.yolov8_blended.640px.aug.v1
3.5m.v3i.yolov8.640px_clahe	       best_e26.pt
3.5m.v3i.yolov8.640px.soil_aug	       best_e50.pt
3.5m.v4i.yolov8.640px		       Inference
3.5m.v4i.yolov8.640px_209	       models
3.5m.v4i.yolov8.640px-2steps.aug2      optuna_yolov8_f1_study.db


In [ ]:
drive_path = '/content/drive/MyDrive/YOLO'
drive_datasets_paths = os.listdir(drive_path)
drive_datasets = len(drive_datasets_paths)
if (drive_datasets) > 1:
    print("There are %d dataset options:" % drive_datasets)
else:
    print("Theres is only 1 dataset:")
drive_datasets_paths

There are 16 dataset options:


['3.5m.v3i.yolov8.640px',
 'Inference',
 'models',
 '3.5m.v3i.yolov8.640px.aug.v1',
 'best_e26.pt',
 'optuna_yolov8_f1_study.db',
 '3.5m.v3i.yolov8.640px.soil_aug',
 '3.5m.v3i.yolov8.640px.aug.v1.soil_aug',
 '3.5m.v3i.yolov8.640px_clahe',
 '3.5m.v4i.yolov8.640px',
 '3.5m.v4i.yolov8_blended.640px',
 '3.5m.v4i.yolov8.640px_209',
 '3.5m.v4i.yolov8_blended.640px.aug.v1',
 'best_e50.pt',
 '3.5m.v4i.yolov8.640px.aug.v1',
 '3.5m.v4i.yolov8.640px-2steps.aug2']

**For this experiments:** `3.5m.v4i.yolov8.640px-2steps.aug2`

In [ ]:
choose_dataset = 16
index = choose_dataset - 1
model_name = os.listdir(drive_path)[index]
print("Chosen model:", model_name)

Chosen model: 3.5m.v4i.yolov8.640px-2steps.aug2


***Readme:***
*   **Option 1:** is desirable if you need to test many subset combinations in the same session (avoid downloading data twice from the cloud).
*   **Option 2:** is desired if you are goint to work just with one dataset  (avoid downloading unnecessary data from the cloud).
*   **Option 3:** is best if you're just going to test one subset combination  (avoid downloading any data from the cloud).



In [ ]:
# Option 2 (download just the needed dataset)
cloud_path = f"/content/drive/MyDrive/YOLO/{model_name}/"
local_path = f"/content/YOLO/"
!mkdir $local_path
!cp -r $cloud_path $local_path

In [ ]:
src_folder = f"/content/YOLO/{model_name}"
data = f"{src_folder}/data.yaml"
data

'/content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/data.yaml'

## Download model

In [ ]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# Load pretrain YOLO v8 model
model = YOLO("yolov8m.pt")

100%|██████████| 49.7M/49.7M [00:00<00:00, 202MB/s]


# Finetuning

### Optimization

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
# Reduce VRAM usage by reducing fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

### Info

In [ ]:
!nvidia-smi

Wed May 14 12:31:28 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!yolo version

8.3.134


# Experiments

In [ ]:
# Set's maximum training time (in hours)
time: float = 0.5 # Depending on remaining Colab cuota
# This overrides the epochs argument, allowing training to automaticallystop after the specified duration.
# Useful for time-constrained training scenarios. (aka Colab cuotas)

## Experiment A 1
### *YOLOv8 Mid | Mix*

- `multi_scale=True`
- `weight_decay=0.0015` (Explorar un - weight_decay ligeramente mayor)
- `momentum=0.98`
- `dropout=0` (default)

**Justificación:** Mantener los parámetros exitosos de Mix 2 y ver si una regularización de weight_decay un poco más fuerte mejora aún más el rendimiento, especialmente en Precision y F½ Score.


### Train

In [ ]:
# Garbage collection
import gc
for i in range(10):
  torch.cuda.empty_cache()
  gc.collect()

In [ ]:
# Train model
model = YOLO("yolov8m.pt")
history = model.train(
    data=data,
    epochs=500,
    val=True,
    imgsz=640,
    batch=-1,
    patience=100,
    time = time,
    multi_scale=True,
    weight_decay=0.0015, # Superior al anterior
    #dropout=0.2,
    momentum=0.98,
)

Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.98, mosaic=1.0, multi_scale=True, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots

100%|██████████| 755k/755k [00:00<00:00, 114MB/s]

Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1      1392  ultralytics.nn.modules.conv.Conv             [3, 48, 3, 2]                 
  1                  -1  1     41664  ultralytics.nn.modules.conv.Conv             [48, 96, 3, 2]                
  2                  -1  2    111360  ultralytics.nn.modules.block.C2f             [96, 96, 2, True]             
  3                  -1  1    166272  ultralytics.nn.modules.conv.Conv             [96, 192, 3, 2]               
  4                  -1  4    813312  ultralytics.nn.modules.block.C2f             [192, 192, 4, True]           
  5                  -1  1    664320  ultralytics.nn.modules.conv.Conv             [192, 384, 3, 2]              
  6                  -1  4   3248640  ultralytics.nn.modules.block.C2f             [384, 384, 4, True]           
  7                  -1  1   1991808  ultralytics

 21                  -1  2   4207104  ultralytics.nn.modules.block.C2f             [960, 576, 2]                 
 22        [15, 18, 21]  1   3776275  ultralytics.nn.modules.head.Detect           [1, [192, 384, 576]]          
Model summary: 169 layers, 25,856,899 parameters, 25,856,883 gradients, 79.1 GFLOPs

Transferred 469/475 items from pretrained weights
Freezing layer 'model.22.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks...


100%|██████████| 5.35M/5.35M [00:00<00:00, 361MB/s]


...

      15/15      14.3G      2.027        1.3      1.372        425        544:  91%|█████████ | 124/136 [01:37<00:09,  1.27it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:01<00:00,  1.76it/s]

                   all        108       3467       0.56      0.533      0.533      0.194



15 epochs completed in 0.501 hours.
Optimizer stripped from runs/detect/train/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/train/weights/best.pt, 52.0MB

Validating runs/detect/train/weights/best.pt...
Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:06<00:00,  2.08s/it]


                   all        108       3467       0.56      0.533      0.533      0.194
Speed: 0.2ms preprocess, 11.5ms inference, 0.0ms loss, 7.6ms postprocess per image
Results saved to runs/detect/train


In [ ]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a2b0c607cd0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.pt',
          data='/content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/data.yaml',
          epochs=500,
          time=0.5,
          patience=100,
          batch=19,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device=None,
          workers=8,
          project=None,
          name='train',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=None,
          multi_scale=True,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.0,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
          

In [ ]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train


### Validation

In [ ]:
# Load currently trained YOLO model
model = YOLO(f"/content/{history.save_dir}/weights/best.pt")

In [ ]:
# Validate the model
results = model.val(data=data,
          batch=64,
          conf=0.224, # best values found with optuna
          iou=0.57, # best values found with optuna
          verbose=True,
          save_json=True)

Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1668.7±788.7 MB/s, size: 74.7 KB)


val: Scanning /content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:07<00:00,  3.69s/it]


                   all        108       3467      0.601      0.538      0.562      0.224
Speed: 4.9ms preprocess, 23.4ms inference, 0.0ms loss, 12.0ms postprocess per image
Saving runs/detect/val/predictions.json...
Results saved to runs/detect/val


In [ ]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val


In [ ]:
save_json(results)

✅ JSON file stored in: runs/detect/val


In [ ]:
matrix = gimme_metrics(results)

Total objects detected: 4532.0
Confusion matrix:
['44.97%', '23.50%']
['31.53%', '0.00%']


### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/saveA/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/saveA/


### Metrics

In [ ]:
matrix

[[2038.0, 1065.0], [1429.0, 0.0]]

In [ ]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 4532.0

Confusion matrix:
[ 44.97% , 23.50% ]
[ 31.53% , 0.00% ]

Metrics:
- Accuracy: 0.450
- Precision: 0.657
- Recall: 0.588
- F1 Score: 0.620
- F½ Score: 0.642
- G-mean: 0.621


Comparación con Reference:

- Accuracy: Mejora (0.448 > 0.406) **+10.34%**
- Precision: Mejora (0.634 > 0.579) **+9.49%**
- Recall: Mejora (0.604 > 0.576)
- F1 Score: Mejora (0.619 > 0.577)
- F½ Score: Mejora (0.628 > 0.578) **+8.65%**
- G-mean: Mejora (0.619 > 0.577)

Hubo mejoras significativas en todas las métricas clave en comparación con el modelo de referencia. Exp A mostró un rendimiento consistentemente superior.


Compararción con Mix 1 (59):

- Accuracy: Mejora (0.448 > 0.433)
- Precision: Mejora (0.634 > 0.611)
- Recall: Constante (0.604 ~ 0.597)
- F1 Score: Mejora (0.619 > 0.604)
- F½ Score: Mejora (0.628 > 0.608)
- G-mean: Mejora (0.619 > 0.604)

Conclusión: Exp A obtuvo mejores resultados que Mix 1 (59) en todas las métricas.

-----
## Experiment B 2
### *YOLOv8 Mid | Mix*
- `multi_scale=True`
- `weight_decay=0.001`
- `momentum=0.99` (Explorar un momentum aún más cercano a 1)
- `dropout=0` (default)

**Justificación:** Mantener el weight_decay de Mix 2 y probar si un momentum extremadamente alto puede ofrecer beneficios adicionales para la convergencia y el rendimiento final en las métricas objetivo.

### Train

In [ ]:
# Garbage collection
import gc
for i in range(10):
  torch.cuda.empty_cache()
  gc.collect()

In [ ]:
# Train model
model = YOLO("yolov8m.pt")
history = model.train(
    data=data,
    epochs=500,
    val=True,
    imgsz=640,
    batch=-1,
    patience=100,
    time = time,
    multi_scale=True,
    weight_decay=0.001,
    #dropout=0.2,
    momentum=0.99, # Superior al anterior
)

Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.99, mosaic=1.0, multi_scale=True, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plot

train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/train/labels.cache... 2570 images, 141 backgrounds, 0 corrupt: 100%|██████████| 2570/2570 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
AutoBatch: Computing optimal batch size for imgsz=640 at 60.0% CUDA memory utilization.
AutoBatch: CUDA:0 (Tesla T4) 14.74G total, 0.35G reserved, 0.32G allocated, 14.07G free
      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output


    25856899       79.07         1.535         36.65         41.82        (1, 3, 640, 640)                    list
    25856899       158.1         2.110         35.49         73.86        (2, 3, 640, 640)                    list
    25856899       316.3         2.949         47.11         79.62        (4, 3, 640, 640)                    list
    25856899       632.5         4.589          82.6         139.8        (8, 3, 640, 640)                    list
    25856899        1265         7.676         155.4         268.7       (16, 3, 640, 640)                    list
AutoBatch: Using batch-size 16 for CUDA:0 8.40G/14.74G (57%) ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1639.8±492.9 MB/s, size: 74.4 KB)


train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/train/labels.cache... 2570 images, 141 backgrounds, 0 corrupt: 100%|██████████| 2570/2570 [00:00<?, ?it/s]

...

      18/18      12.2G       2.02       1.35      1.473        280        832:  38%|███▊      | 61/161 [00:39<01:04,  1.55it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:01<00:00,  2.08it/s]

                   all        108       3467      0.533      0.517      0.508      0.186



18 epochs completed in 0.501 hours.
Optimizer stripped from runs/detect/train2/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/train2/weights/best.pt, 52.0MB

Validating runs/detect/train2/weights/best.pt...
Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:06<00:00,  1.53s/it]


                   all        108       3467      0.554      0.528      0.524      0.196
Speed: 0.3ms preprocess, 11.8ms inference, 0.0ms loss, 3.8ms postprocess per image
Results saved to runs/detect/train2


In [ ]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a2b0c6ce210>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.pt',
          data='/content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/data.yaml',
          epochs=500,
          time=0.5,
          patience=100,
          batch=16,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device=None,
          workers=8,
          project=None,
          name='train2',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=None,
          multi_scale=True,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.0,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
         

In [ ]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train2


### Validation

In [ ]:
# Load currently trained YOLO model
model = YOLO(f"/content/{history.save_dir}/weights/best.pt")

In [ ]:
# Validate the model
results = model.val(data=data,
          batch=64,
          conf=0.224, # best values found with optuna
          iou=0.57, # best values found with optuna
          verbose=True,
          save_json=True)

Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2236.3±396.5 MB/s, size: 89.8 KB)


val: Scanning /content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.33s/it]


                   all        108       3467       0.61       0.55      0.565      0.229
Speed: 5.1ms preprocess, 23.8ms inference, 0.0ms loss, 2.3ms postprocess per image
Saving runs/detect/val2/predictions.json...
Results saved to runs/detect/val2


In [ ]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val2


In [ ]:
save_json(results)

✅ JSON file stored in: runs/detect/val2


In [ ]:
matrix = gimme_metrics(results)

Total objects detected: 4523.0
Confusion matrix:
['45.74%', '23.35%']
['30.91%', '0.00%']


### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/saveB/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/saveB/


### Metrics

In [ ]:
matrix

[[2069.0, 1056.0], [1398.0, 0.0]]

In [ ]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 4523.0

Confusion matrix:
[ 45.74% , 23.35% ]
[ 30.91% , 0.00% ]

Metrics:
- Accuracy: 0.457
- Precision: 0.662
- Recall: 0.597
- F1 Score: 0.628
- F½ Score: 0.648
- G-mean: 0.629


Comparación con Reference:

- Accuracy: Mejora (0.418 > 0.406)
- Precision: Mejora (0.633 > 0.579) **+9.33%**
- Recall: Empeoramiento (0.552 < 0.576) **-4.17%**
- F1 Score: Constante (0.590 ~ 0.577)
- F½ Score: Mejora (0.615 > 0.578) **+6.40%**
- G-mean: Constante (0.591 ~ 0.577)

Compararción con Mix 1 (59):

- Accuracy: Empeora (0.418 < 0.433)
- Precision: Mejora (0.633 > 0.611)
- Recall: Empeora (0.552 < 0.597) **-7.54%**
- F1 Score: Empeora (0.590 < 0.604)
- F½ Score: Mejora (0.615 > 0.608)
- G-mean: Mejora (0.604 > 0.591)

-----
## Experiment C 3
### *YOLOv8 Mid | Mix*

Best mix + dropout

### Train

In [ ]:
# Garbage collection
import gc
for i in range(10):
  torch.cuda.empty_cache()
  gc.collect()

In [ ]:
# Train model
model = YOLO("yolov8m.pt")
history = model.train(
    data=data,
    epochs=500,
    val=True,
    imgsz=640,
    batch=-1,
    patience=100,
    time = time,
    multi_scale=True,
    weight_decay=0.001,
    dropout=0.2,
    momentum=0.99, # Superior al anterior
)

Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.2, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.99, mosaic=1.0, multi_scale=True, name=train3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plot

train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/train/labels.cache... 2570 images, 141 backgrounds, 0 corrupt: 100%|██████████| 2570/2570 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
AutoBatch: Computing optimal batch size for imgsz=640 at 60.0% CUDA memory utilization.
AutoBatch: CUDA:0 (Tesla T4) 14.74G total, 0.36G reserved, 0.32G allocated, 14.06G free


      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output
    25856899       79.07         1.535         40.87         42.45        (1, 3, 640, 640)                    list
    25856899       158.1         2.070          35.7         73.87        (2, 3, 640, 640)                    list
    25856899       316.3         2.928         45.44         82.66        (4, 3, 640, 640)                    list
    25856899       632.5         4.507         82.57         140.1        (8, 3, 640, 640)                    list
    25856899        1265         7.636         155.9         272.9       (16, 3, 640, 640)                    list
AutoBatch: Using batch-size 16 for CUDA:0 8.35G/14.74G (57%) ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1439.4±396.6 MB/s, size: 74.4 KB)


train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/train/labels.cache... 2570 images, 141 backgrounds, 0 corrupt: 100%|██████████| 2570/2570 [00:00<?, ?it/s]

...

      18/18      12.2G      2.024       1.35      1.472        320        864:  48%|████▊     | 77/161 [00:48<00:53,  1.58it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:02<00:00,  1.77it/s]

                   all        108       3467      0.534      0.515      0.513      0.191



18 epochs completed in 0.501 hours.
Optimizer stripped from runs/detect/train3/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/train3/weights/best.pt, 52.0MB

Validating runs/detect/train3/weights/best.pt...
Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:06<00:00,  1.56s/it]


                   all        108       3467      0.554      0.528      0.524      0.196
Speed: 0.2ms preprocess, 11.6ms inference, 0.0ms loss, 5.2ms postprocess per image
Results saved to runs/detect/train3


In [ ]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a2b0d441b50>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.pt',
          data='/content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/data.yaml',
          epochs=500,
          time=0.5,
          patience=100,
          batch=16,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device=None,
          workers=8,
          project=None,
          name='train3',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=None,
          multi_scale=True,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.2,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
         

In [ ]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train3


### Validation

In [ ]:
# Load currently trained YOLO model
model = YOLO(f"/content/{history.save_dir}/weights/best.pt")

In [ ]:
# Validate the model
results = model.val(data=data,
          batch=64,
          conf=0.224, # best values found with optuna
          iou=0.57, # best values found with optuna
          verbose=True,
          save_json=True)

Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2092.5±256.0 MB/s, size: 93.0 KB)


val: Scanning /content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.44s/it]


                   all        108       3467       0.61       0.55      0.565      0.229
Speed: 0.3ms preprocess, 27.8ms inference, 0.0ms loss, 2.2ms postprocess per image
Saving runs/detect/val3/predictions.json...
Results saved to runs/detect/val3


In [ ]:
print(f"Saved into: {results.save_dir}")

In [ ]:
save_json(results)

✅ JSON file stored in: runs/detect/val3


In [ ]:
matrix = gimme_metrics(results)

Total objects detected: 4523.0
Confusion matrix:
['45.74%', '23.35%']
['30.91%', '0.00%']


### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/saveC/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/saveC/


### Metrics

In [ ]:
matrix

In [ ]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 4523.0

Confusion matrix:
[ 45.74% , 23.35% ]
[ 30.91% , 0.00% ]

Metrics:
- Accuracy: 0.457
- Precision: 0.662
- Recall: 0.597
- F1 Score: 0.628
- F½ Score: 0.648
- G-mean: 0.629


Comparación de EXP C con Reference (Base = Reference)

Accuracy: ((0.429 - 0.406) / 0.406) * 100 ≈ +5.66% (Mejora)
Precision: ((0.660 - 0.579) / 0.579) * 100 ≈ +13.99% (Mejora)
Recall: ((0.550 - 0.576) / 0.576) * 100 ≈ -4.51% (Empeoramiento)
F1 Score: ((0.600 - 0.577) / 0.577) * 100 ≈ +3.99% (Mejora)
F½ Score: ((0.634 - 0.578) / 0.578) * 100 ≈ +9.69% (Mejora)
G-mean: ((0.603 - 0.577) / 0.577) * 100 ≈ +4.51% (Mejora)
Conclusión vs Reference: EXP C mostró mejoras significativas en Accuracy, Precision, F1 Score, F½ Score y G-mean en comparación con la Referencia, pero tuvo un empeoramiento en Recall.

Comparación de EXP C con Mix 1 (59) (Base = Mix 1)

Accuracy: ((0.429 - 0.433) / 0.433) * 100 ≈ -0.92% (Constante, variación < 2%)
Precision: ((0.660 - 0.611) / 0.611) * 100 ≈ +7.97% (Mejora)
Recall: ((0.550 - 0.597) / 0.597) * 100 ≈ -7.87% (Empeoramiento)
F1 Score: ((0.600 - 0.604) / 0.604) * 100 ≈ -0.66% (Constante, variación < 2%)
F½ Score: ((0.634 - 0.608) / 0.608) * 100 ≈ +4.28% (Mejora)
G-mean: ((0.603 - 0.604) / 0.604) * 100 ≈ -0.17% (Constante, variación < 2%)


-----
## Experiment D 4
### *YOLOv8 Mid | Mix*

Best mix w/o dropout

### Train

In [ ]:
# Garbage collection
import gc
for i in range(10):
  torch.cuda.empty_cache()
  gc.collect()

In [ ]:
# Set's maximum training time (in hours)
time: float = 1 # Depending on remaining Colab cuota
# This overrides the epochs argument, allowing training to automaticallystop after the specified duration.
# Useful for time-constrained training scenarios. (aka Colab cuotas)

In [ ]:
# Train model
model = YOLO("yolov8m.pt")
history = model.train(
    data=data,
    epochs=500,
    val=True,
    imgsz=640,
    batch=-1,
    patience=100,
    time = time,
    multi_scale=True,
    weight_decay=0.0015, # Superior al anterior
    dropout=0.2,
    momentum=0.99, # Superior al anterior
)

Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.2, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.99, mosaic=1.0, multi_scale=True, name=train4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plot

train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/train/labels.cache... 2570 images, 141 backgrounds, 0 corrupt: 100%|██████████| 2570/2570 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
AutoBatch: Computing optimal batch size for imgsz=640 at 60.0% CUDA memory utilization.
AutoBatch: CUDA:0 (Tesla T4) 14.74G total, 0.36G reserved, 0.32G allocated, 14.06G free
      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output


    25856899       79.07         1.535         37.95         42.58        (1, 3, 640, 640)                    list
    25856899       158.1         2.091         35.82          73.8        (2, 3, 640, 640)                    list
    25856899       316.3         2.923         46.88         77.43        (4, 3, 640, 640)                    list
    25856899       632.5         4.547         81.54         141.5        (8, 3, 640, 640)                    list
    25856899        1265         7.657         155.6         271.2       (16, 3, 640, 640)                    list
AutoBatch: Using batch-size 16 for CUDA:0 8.38G/14.74G (57%) ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1327.7±400.2 MB/s, size: 74.4 KB)


train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/train/labels.cache... 2570 images, 141 backgrounds, 0 corrupt: 100%|██████████| 2570/2570 [00:00<?, ?it/s]

...

      35/35      12.2G       1.91      1.204      1.388        231        352:  32%|███▏      | 52/161 [00:33<01:10,  1.55it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:01<00:00,  2.35it/s]

                   all        108       3467      0.585      0.526      0.545      0.199



35 epochs completed in 1.001 hours.
Optimizer stripped from runs/detect/train4/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/train4/weights/best.pt, 52.0MB

Validating runs/detect/train4/weights/best.pt...
Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:06<00:00,  1.68s/it]


                   all        108       3467      0.585       0.54       0.55        0.2
Speed: 0.5ms preprocess, 11.1ms inference, 0.0ms loss, 7.7ms postprocess per image
Results saved to runs/detect/train4


In [ ]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a2a9f517210>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.pt',
          data='/content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/data.yaml',
          epochs=500,
          time=1,
          patience=100,
          batch=16,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device=None,
          workers=8,
          project=None,
          name='train4',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=None,
          multi_scale=True,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.2,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
          i

In [ ]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train4


### Validation

In [ ]:
# Load currently trained YOLO model
model = YOLO(f"/content/{history.save_dir}/weights/best.pt")

In [ ]:
# Validate the model
results = model.val(data=data,
          batch=64,
          conf=0.224, # best values found with optuna
          iou=0.57, # best values found with optuna
          verbose=True,
          save_json=True)

Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2081.7±990.9 MB/s, size: 89.8 KB)


val: Scanning /content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.46s/it]


                   all        108       3467      0.632      0.539      0.578      0.232
Speed: 0.3ms preprocess, 27.5ms inference, 0.0ms loss, 3.0ms postprocess per image
Saving runs/detect/val4/predictions.json...
Results saved to runs/detect/val4


In [ ]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val4


In [ ]:
save_json(results)

✅ JSON file stored in: runs/detect/val4


In [ ]:
matrix = gimme_metrics(results)

Total objects detected: 4389.0
Confusion matrix:
['46.41%', '21.01%']
['32.58%', '0.00%']


### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/saveD/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/saveD/


### Metrics

In [ ]:
matrix

[[2037.0, 922.0], [1430.0, 0.0]]

In [ ]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Comparación con EXP C:
- Accuracy: Constante (0.424 ~ 0.429)
- Precision: Empeora (0.633 < 0.660) **-4.09%**
- Recall: Constante (0.562 ~ 0.550)
- F1 Score: Constante (0.595 ~ 0.600)
- F½ Score: Empeora (0.617 < 0.634)
- G-mean: Constante (0.597 ~ 0.603)

Podemos apreciar el impacto negativo de haber removido el dropout, observamos que EXP D tuvo un empeoramiento notable en Precision y F½ Score. Sin embargo, mostró una ligera mejora en Recall (+2.18%).

Comparación con Mix 3 (61):
- Accuracy: Constante (0.424 vs 0.433)
- Precision: Constante (0.633 vs 0.636
- Recall: Empeora (0.562 vs 0.576) -2.43%
- F1 Score: Constante (0.595 vs 0.605)
- F½ Score: Constante (0.617 vs 0.623)
- G-mean: Constante (0.597 vs 0.605)

**Conclusión:**  Podemos concluir que agregar un dropout pequeño mejora el entrenamiento.

-----
## Experiment E 5
### *YOLOv8 Mid | Mix*

Best mix w/o dropout

### Train

In [ ]:
# Garbage collection
import gc
for i in range(10):
  torch.cuda.empty_cache()
  gc.collect()

In [ ]:
# Set's maximum training time (in hours)
time: float = 1 # Depending on remaining Colab cuota
# This overrides the epochs argument, allowing training to automaticallystop after the specified duration.
# Useful for time-constrained training scenarios. (aka Colab cuotas)

In [ ]:
# Train model
model = YOLO("yolov8m.pt")
history = model.train(
    data=data,
    epochs=500,
    val=True,
    imgsz=640,
    batch=-1,
    patience=100,
    time = time,
    multi_scale=True,
    weight_decay=0.0015, # Superior al anterior
    dropout=0.1,
    momentum=0.99, # Superior al anterior
)

In [ ]:
history

In [ ]:
# Show the hyperparameters set
model.trainer.validator.args

In [ ]:
print(f"Saved into: {history.save_dir}")

### Validation

In [ ]:
# Load currently trained YOLO model
model = YOLO(f"/content/{history.save_dir}/weights/best.pt")

In [ ]:
# Validate the model
results = model.val(data=data,
          batch=64,
          conf=0.224, # best values found with optuna
          iou=0.57, # best values found with optuna
          verbose=True,
          save_json=True)

In [ ]:
print(f"Saved into: {results.save_dir}")

In [ ]:
save_json(results)

In [ ]:
matrix = gimme_metrics(results)

### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/saveE/')

### Metrics

In [ ]:
matrix

In [ ]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Comparación con EXP C:
- Accuracy: Constante (0.424 ~ 0.429)
- Precision: Empeora (0.633 < 0.660) **-4.09%**
- Recall: Constante (0.562 ~ 0.550)
- F1 Score: Constante (0.595 ~ 0.600)
- F½ Score: Empeora (0.617 < 0.634)
- G-mean: Constante (0.597 ~ 0.603)

Podemos apreciar el impacto negativo de haber removido el dropout, observamos que EXP D tuvo un empeoramiento notable en Precision y F½ Score. Sin embargo, mostró una ligera mejora en Recall (+2.18%).

Comparación con Mix 3 (61):
- Accuracy: Constante (0.424 vs 0.433)
- Precision: Constante (0.633 vs 0.636
- Recall: Empeora (0.562 vs 0.576) -2.43%
- F1 Score: Constante (0.595 vs 0.605)
- F½ Score: Constante (0.617 vs 0.623)
- G-mean: Constante (0.597 vs 0.605)

**Conclusión:**  Podemos concluir que agregar un dropout pequeño mejora el entrenamiento.

-----
## Experiment F 6
### *YOLOv8 Mid | Mix*

Best mix w/o dropout

### Train

In [ ]:
# Garbage collection
import gc
for i in range(10):
  torch.cuda.empty_cache()
  gc.collect()

In [ ]:
# Train model
model = YOLO("yolov8m.pt")
history = model.train(
    data=data,
    epochs=500,
    val=True,
    imgsz=640,
    batch=-1,
    patience=100,
    time = time,
    multi_scale=True,
    weight_decay=0.002, # Superior al anterior
    dropout=0.1,
    momentum=0.99,
)

Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.99, mosaic=1.0, multi_scale=True, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots

train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/train/labels.cache... 2570 images, 141 backgrounds, 0 corrupt: 100%|██████████| 2570/2570 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
AutoBatch: Computing optimal batch size for imgsz=640 at 60.0% CUDA memory utilization.
AutoBatch: CUDA:0 (Tesla T4) 14.74G total, 0.60G reserved, 0.51G allocated, 13.62G free
      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output


    25856899       79.07         1.579         37.76          41.3        (1, 3, 640, 640)                    list
    25856899       158.1         2.091          34.3         68.85        (2, 3, 640, 640)                    list
    25856899       316.3         2.928         46.46         85.21        (4, 3, 640, 640)                    list
    25856899       632.5         4.463         79.12         132.2        (8, 3, 640, 640)                    list
    25856899        1265         7.510         149.3           256       (16, 3, 640, 640)                    list
AutoBatch: Using batch-size 17 for CUDA:0 9.06G/14.74G (61%) ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1672.4±510.4 MB/s, size: 74.4 KB)


train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/train/labels.cache... 2570 images, 141 backgrounds, 0 corrupt: 100%|██████████| 2570/2570 [00:00<?, ?it/s]

...

      18/18        13G      2.041      1.371      1.497        329        640:  39%|███▉      | 59/152 [00:42<01:06,  1.40it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:01<00:00,  2.41it/s]

                   all        108       3467      0.534      0.521      0.515      0.192



18 epochs completed in 0.501 hours.
Optimizer stripped from runs/detect/train/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/train/weights/best.pt, 52.0MB

Validating runs/detect/train/weights/best.pt...
Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:05<00:00,  1.45s/it]


                   all        108       3467      0.533      0.521      0.515      0.192
Speed: 0.3ms preprocess, 11.5ms inference, 0.0ms loss, 7.6ms postprocess per image
Results saved to runs/detect/train


In [ ]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e4b2852f750>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.pt',
          data='/content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/data.yaml',
          epochs=500,
          time=0.5,
          patience=100,
          batch=17,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device=None,
          workers=8,
          project=None,
          name='train',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=None,
          multi_scale=True,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.1,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
          

In [ ]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train


### Validation

In [ ]:
# Load currently trained YOLO model
model = YOLO(f"/content/{history.save_dir}/weights/best.pt")

In [ ]:
# Validate the model
results = model.val(data=data,
          batch=64,
          conf=0.224, # best values found with optuna
          iou=0.57, # best values found with optuna
          verbose=True,
          save_json=True)

Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1874.5±799.4 MB/s, size: 96.0 KB)


val: Scanning /content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:07<00:00,  3.70s/it]


                   all        108       3467      0.631      0.494       0.55      0.224
Speed: 0.2ms preprocess, 31.8ms inference, 0.0ms loss, 4.4ms postprocess per image
Saving runs/detect/val/predictions.json...
Results saved to runs/detect/val


In [ ]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val


In [ ]:
save_json(results)

✅ JSON file stored in: runs/detect/val


In [ ]:
matrix = gimme_metrics(results)

Total objects detected: 4315.0
Confusion matrix:
['43.29%', '19.65%']
['37.06%', '0.00%']


### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/saveE/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/saveE/


### Metrics

In [ ]:
matrix

[[1868.0, 848.0], [1599.0, 0.0]]

In [ ]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 4315.0

Confusion matrix:
[ 43.29% , 19.65% ]
[ 37.06% , 0.00% ]

Metrics:
- Accuracy: 0.433
- Precision: 0.688
- Recall: 0.539
- F1 Score: 0.604
- F½ Score: 0.652
- G-mean: 0.609


# Ref E

Confusion matrix:
[ 48.77% , 23.04% ]
[ 28.19% , 0.00% ]

Metrics:
- Accuracy: 0.488
- Precision: 0.679
- Recall: 0.634
- F1 Score: 0.656
- F½ Score: 0.670
- G-mean: 0.656

Comparación con EXP C:
- Accuracy: Constante (0.424 ~ 0.429)
- Precision: Empeora (0.633 < 0.660) **-4.09%**
- Recall: Constante (0.562 ~ 0.550)
- F1 Score: Constante (0.595 ~ 0.600)
- F½ Score: Empeora (0.617 < 0.634)
- G-mean: Constante (0.597 ~ 0.603)

Podemos apreciar el impacto negativo de haber removido el dropout, observamos que EXP D tuvo un empeoramiento notable en Precision y F½ Score. Sin embargo, mostró una ligera mejora en Recall (+2.18%).

Comparación con Mix 3 (61):
- Accuracy: Constante (0.424 vs 0.433)
- Precision: Constante (0.633 vs 0.636
- Recall: Empeora (0.562 vs 0.576) -2.43%
- F1 Score: Constante (0.595 vs 0.605)
- F½ Score: Constante (0.617 vs 0.623)
- G-mean: Constante (0.597 vs 0.605)

**Conclusión:**  Podemos concluir que agregar un dropout pequeño mejora el entrenamiento.

-----
## Experiment G 7
### *YOLOv8 Mid | Mix*

Best mix w/o dropout

### Train

In [ ]:
# Garbage collection
import gc
for i in range(10):
  torch.cuda.empty_cache()
  gc.collect()

In [ ]:
 # Train model
model = YOLO("yolov8m.pt")
history = model.train(
    data=data,
    epochs=500,
    val=True,
    imgsz=640,
    batch=-1,
    patience=100,
    time = time,
    multi_scale=True,
    weight_decay=0.0015,
    #dropout=0.1, # Sin dropout
    momentum=0.99,
)

Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.99, mosaic=1.0, multi_scale=True, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plot

train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/train/labels.cache... 2570 images, 141 backgrounds, 0 corrupt: 100%|██████████| 2570/2570 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
AutoBatch: Computing optimal batch size for imgsz=640 at 60.0% CUDA memory utilization.
AutoBatch: CUDA:0 (Tesla T4) 14.74G total, 0.35G reserved, 0.32G allocated, 14.07G free
      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output


    25856899       79.07         1.531         34.32         41.43        (1, 3, 640, 640)                    list
    25856899       158.1         2.068         35.62         73.15        (2, 3, 640, 640)                    list
    25856899       316.3         2.926         46.78         80.44        (4, 3, 640, 640)                    list
    25856899       632.5         4.507          80.9         136.4        (8, 3, 640, 640)                    list
    25856899        1265         7.573           152           263       (16, 3, 640, 640)                    list
AutoBatch: Using batch-size 16 for CUDA:0 8.30G/14.74G (56%) ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1926.3±612.9 MB/s, size: 74.4 KB)


train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/train/labels.cache... 2570 images, 141 backgrounds, 0 corrupt: 100%|██████████| 2570/2570 [00:00<?, ?it/s]

...

      18/18      12.2G      2.016      1.357      1.477        280        832:  38%|███▊      | 61/161 [00:39<01:04,  1.55it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:01<00:00,  2.36it/s]

                   all        108       3467      0.552      0.496      0.509      0.187



18 epochs completed in 0.501 hours.
Optimizer stripped from runs/detect/train2/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/train2/weights/best.pt, 52.0MB

Validating runs/detect/train2/weights/best.pt...
Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:06<00:00,  1.68s/it]


                   all        108       3467      0.547      0.518      0.517      0.196
Speed: 0.5ms preprocess, 11.2ms inference, 0.0ms loss, 6.9ms postprocess per image
Results saved to runs/detect/train2


In [ ]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e4ad01a2210>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.pt',
          data='/content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/data.yaml',
          epochs=500,
          time=0.5,
          patience=100,
          batch=16,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device=None,
          workers=8,
          project=None,
          name='train2',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=None,
          multi_scale=True,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.0,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
         

In [ ]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train2


### Validation

In [ ]:
# Load currently trained YOLO model
model = YOLO(f"/content/{history.save_dir}/weights/best.pt")

In [ ]:
# Validate the model
results = model.val(data=data,
          batch=64,
          conf=0.224, # best values found with optuna
          iou=0.57, # best values found with optuna
          verbose=True,
          save_json=True)

Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1600.6±778.7 MB/s, size: 89.8 KB)


val: Scanning /content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.39s/it]


                   all        108       3467      0.651      0.496      0.559      0.235
Speed: 8.1ms preprocess, 23.5ms inference, 0.0ms loss, 2.5ms postprocess per image
Saving runs/detect/val2/predictions.json...
Results saved to runs/detect/val2


In [ ]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val2


In [ ]:
save_json(results)

✅ JSON file stored in: runs/detect/val2


In [ ]:
matrix = gimme_metrics(results)

Total objects detected: 4230.0
Confusion matrix:
['44.42%', '18.04%']
['37.54%', '0.00%']


### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/saveG/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/saveG/


### Metrics

In [ ]:
matrix

[[1879.0, 763.0], [1588.0, 0.0]]

In [ ]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 4230.0

Confusion matrix:
[ 44.42% , 18.04% ]
[ 37.54% , 0.00% ]

Metrics:
- Accuracy: 0.444
- Precision: 0.711
- Recall: 0.542
- F1 Score: 0.615
- F½ Score: 0.669
- G-mean: 0.621


Comparación con EXP C:
- Accuracy: Constante (0.424 ~ 0.429)
- Precision: Empeora (0.633 < 0.660) **-4.09%**
- Recall: Constante (0.562 ~ 0.550)
- F1 Score: Constante (0.595 ~ 0.600)
- F½ Score: Empeora (0.617 < 0.634)
- G-mean: Constante (0.597 ~ 0.603)

Podemos apreciar el impacto negativo de haber removido el dropout, observamos que EXP D tuvo un empeoramiento notable en Precision y F½ Score. Sin embargo, mostró una ligera mejora en Recall (+2.18%).

Comparación con Mix 3 (61):
- Accuracy: Constante (0.424 vs 0.433)
- Precision: Constante (0.633 vs 0.636
- Recall: Empeora (0.562 vs 0.576) -2.43%
- F1 Score: Constante (0.595 vs 0.605)
- F½ Score: Constante (0.617 vs 0.623)
- G-mean: Constante (0.597 vs 0.605)

**Conclusión:**  Podemos concluir que agregar un dropout pequeño mejora el entrenamiento.

-----
## Experiment H 8
### *YOLOv8 Mid | Mix*

Best mix w/o dropout

### Train

In [ ]:
# Garbage collection
import gc
for i in range(10):
  torch.cuda.empty_cache()
  gc.collect()

In [ ]:
# Train model
model = YOLO("yolov8m.pt")
history = model.train(
    data=data,
    epochs=500,
    val=True,
    imgsz=640,
    batch=-1,
    patience=100,
    time = time,
    multi_scale=True,
    weight_decay=0.002,
    #dropout=0.1, # Sin dropout
    momentum=0.99,
)

Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.99, mosaic=1.0, multi_scale=True, name=train3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plot

train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/train/labels.cache... 2570 images, 141 backgrounds, 0 corrupt: 100%|██████████| 2570/2570 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
AutoBatch: Computing optimal batch size for imgsz=640 at 60.0% CUDA memory utilization.
AutoBatch: CUDA:0 (Tesla T4) 14.74G total, 0.36G reserved, 0.32G allocated, 14.06G free
      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output


    25856899       79.07         1.533         35.58         41.62        (1, 3, 640, 640)                    list
    25856899       158.1         2.089         35.49          73.4        (2, 3, 640, 640)                    list
    25856899       316.3         2.926         47.21         84.99        (4, 3, 640, 640)                    list
    25856899       632.5         4.505         79.72         135.4        (8, 3, 640, 640)                    list
    25856899        1265         7.571           151         259.3       (16, 3, 640, 640)                    list
AutoBatch: Using batch-size 16 for CUDA:0 8.30G/14.74G (56%) ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1844.6±481.0 MB/s, size: 74.4 KB)


train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/train/labels.cache... 2570 images, 141 backgrounds, 0 corrupt: 100%|██████████| 2570/2570 [00:00<?, ?it/s]

...

      18/18      12.2G       2.03      1.346      1.459        210        416:  28%|██▊       | 45/161 [00:27<01:10,  1.65it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:01<00:00,  2.38it/s]

                   all        108       3467      0.521      0.487      0.481      0.174



18 epochs completed in 0.501 hours.
Optimizer stripped from runs/detect/train3/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/train3/weights/best.pt, 52.0MB

Validating runs/detect/train3/weights/best.pt...
Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:05<00:00,  1.49s/it]


                   all        108       3467      0.557      0.506      0.515      0.195
Speed: 0.5ms preprocess, 11.9ms inference, 0.0ms loss, 5.8ms postprocess per image
Results saved to runs/detect/train3


In [ ]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e4ae7f5fc10>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.pt',
          data='/content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/data.yaml',
          epochs=500,
          time=0.5,
          patience=100,
          batch=16,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device=None,
          workers=8,
          project=None,
          name='train3',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=None,
          multi_scale=True,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.0,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
         

In [ ]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train3


### Validation

In [ ]:
# Load currently trained YOLO model
model = YOLO(f"/content/{history.save_dir}/weights/best.pt")

In [ ]:
# Validate the model
results = model.val(data=data,
          batch=64,
          conf=0.224, # best values found with optuna
          iou=0.57, # best values found with optuna
          verbose=True,
          save_json=True)

Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1870.2±640.1 MB/s, size: 87.8 KB)


val: Scanning /content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.20s/it]


                   all        108       3467      0.621      0.518      0.558      0.228
Speed: 6.1ms preprocess, 23.6ms inference, 0.0ms loss, 2.2ms postprocess per image
Saving runs/detect/val3/predictions.json...
Results saved to runs/detect/val3


In [ ]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val3


In [ ]:
save_json(results)

✅ JSON file stored in: runs/detect/val3


In [ ]:
matrix = gimme_metrics(results)

Total objects detected: 4418.0
Confusion matrix:
['43.87%', '21.53%']
['34.61%', '0.00%']


### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/saveG/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/saveG/


### Metrics

In [ ]:
matrix

[[1938.0, 951.0], [1529.0, 0.0]]

In [ ]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 4418.0

Confusion matrix:
[ 43.87% , 21.53% ]
[ 34.61% , 0.00% ]

Metrics:
- Accuracy: 0.439
- Precision: 0.671
- Recall: 0.559
- F1 Score: 0.610
- F½ Score: 0.645
- G-mean: 0.612


Comparación con EXP C:
- Accuracy: Constante (0.424 ~ 0.429)
- Precision: Empeora (0.633 < 0.660) **-4.09%**
- Recall: Constante (0.562 ~ 0.550)
- F1 Score: Constante (0.595 ~ 0.600)
- F½ Score: Empeora (0.617 < 0.634)
- G-mean: Constante (0.597 ~ 0.603)

Podemos apreciar el impacto negativo de haber removido el dropout, observamos que EXP D tuvo un empeoramiento notable en Precision y F½ Score. Sin embargo, mostró una ligera mejora en Recall (+2.18%).

Comparación con Mix 3 (61):
- Accuracy: Constante (0.424 vs 0.433)
- Precision: Constante (0.633 vs 0.636
- Recall: Empeora (0.562 vs 0.576) -2.43%
- F1 Score: Constante (0.595 vs 0.605)
- F½ Score: Constante (0.617 vs 0.623)
- G-mean: Constante (0.597 vs 0.605)

**Conclusión:**  Podemos concluir que agregar un dropout pequeño mejora el entrenamiento.

-----
## Experiment E 2.0 9
### *YOLOv8 Mid | Mix*

Se repite entrenamiento anterior para que tenga el mismo tiempode entrenamiento y poder compararlo.

### Train

In [ ]:
# Garbage collection
import gc
for i in range(10):
  torch.cuda.empty_cache()
  gc.collect()

In [ ]:
# Train model
model = YOLO("yolov8m.pt")
history = model.train(
    data=data,
    epochs=500,
    val=True,
    imgsz=640,
    batch=-1,
    patience=100,
    time = time,
    multi_scale=True,
    weight_decay=0.0015, # Superior al anterior
    dropout=0.1,
    momentum=0.99, # Superior al anterior
)

Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.99, mosaic=1.0, multi_scale=True, name=train4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plot

train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/train/labels.cache... 2570 images, 141 backgrounds, 0 corrupt: 100%|██████████| 2570/2570 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
AutoBatch: Computing optimal batch size for imgsz=640 at 60.0% CUDA memory utilization.
AutoBatch: CUDA:0 (Tesla T4) 14.74G total, 0.35G reserved, 0.32G allocated, 14.07G free


      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output
    25856899       79.07         1.533         69.49         89.11        (1, 3, 640, 640)                    list
    25856899       158.1         2.089          58.2         75.23        (2, 3, 640, 640)                    list
    25856899       316.3         2.926         54.23         90.19        (4, 3, 640, 640)                    list
    25856899       632.5         4.507         81.37         138.6        (8, 3, 640, 640)                    list
    25856899        1265         7.571         154.6         267.6       (16, 3, 640, 640)                    list
AutoBatch: Using batch-size 16 for CUDA:0 8.29G/14.74G (56%) ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1716.7±407.8 MB/s, size: 74.4 KB)


train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/train/labels.cache... 2570 images, 141 backgrounds, 0 corrupt: 100%|██████████| 2570/2570 [00:00<?, ?it/s]

...

      17/17      12.6G      2.035      1.346      1.452        187        448:  98%|█████████▊| 158/161 [01:40<00:01,  1.57it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:02<00:00,  1.78it/s]

                   all        108       3467      0.542      0.513       0.52      0.196



17 epochs completed in 0.504 hours.
Optimizer stripped from runs/detect/train4/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/train4/weights/best.pt, 52.0MB

Validating runs/detect/train4/weights/best.pt...
Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:07<00:00,  1.84s/it]


                   all        108       3467      0.545      0.512       0.52      0.195
Speed: 0.3ms preprocess, 11.0ms inference, 0.0ms loss, 8.4ms postprocess per image
Results saved to runs/detect/train4


In [ ]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e4ad0133c50>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.pt',
          data='/content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/data.yaml',
          epochs=500,
          time=0.5,
          patience=100,
          batch=16,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device=None,
          workers=8,
          project=None,
          name='train4',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=None,
          multi_scale=True,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.1,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
         

In [ ]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train4


### Validation

In [ ]:
# Load currently trained YOLO model
model = YOLO(f"/content/{history.save_dir}/weights/best.pt")

In [ ]:
# Validate the model
results = model.val(data=data,
          batch=64,
          conf=0.224, # best values found with optuna
          iou=0.57, # best values found with optuna
          verbose=True,
          save_json=True)

Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1704.8±536.5 MB/s, size: 93.6 KB)


val: Scanning /content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:07<00:00,  3.57s/it]


                   all        108       3467      0.634        0.5      0.561       0.23
Speed: 0.3ms preprocess, 30.9ms inference, 0.0ms loss, 3.6ms postprocess per image
Saving runs/detect/val4/predictions.json...
Results saved to runs/detect/val4


In [ ]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val4


In [ ]:
save_json(results)

✅ JSON file stored in: runs/detect/val4


In [ ]:
matrix = gimme_metrics(results)

Total objects detected: 4323.0
Confusion matrix:
['43.42%', '19.80%']
['36.78%', '0.00%']


### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/saveE2/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/saveE2/


### Metrics

In [ ]:
matrix

[[1877.0, 856.0], [1590.0, 0.0]]

In [ ]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 4323.0

Confusion matrix:
[ 43.42% , 19.80% ]
[ 36.78% , 0.00% ]

Metrics:
- Accuracy: 0.434
- Precision: 0.687
- Recall: 0.541
- F1 Score: 0.605
- F½ Score: 0.652
- G-mean: 0.610


Comparación con EXP C:
- Accuracy: Constante (0.424 ~ 0.429)
- Precision: Empeora (0.633 < 0.660) **-4.09%**
- Recall: Constante (0.562 ~ 0.550)
- F1 Score: Constante (0.595 ~ 0.600)
- F½ Score: Empeora (0.617 < 0.634)
- G-mean: Constante (0.597 ~ 0.603)

Podemos apreciar el impacto negativo de haber removido el dropout, observamos que EXP D tuvo un empeoramiento notable en Precision y F½ Score. Sin embargo, mostró una ligera mejora en Recall (+2.18%).

Comparación con Mix 3 (61):
- Accuracy: Constante (0.424 vs 0.433)
- Precision: Constante (0.633 vs 0.636
- Recall: Empeora (0.562 vs 0.576) -2.43%
- F1 Score: Constante (0.595 vs 0.605)
- F½ Score: Constante (0.617 vs 0.623)
- G-mean: Constante (0.597 vs 0.605)

**Conclusión:**  Podemos concluir que agregar un dropout pequeño mejora el entrenamiento.

-----
## Experiment I 10
### *YOLOv8 Mid | Mix*

Se repite entrenamiento anterior para que tenga el mismo tiempode entrenamiento y poder compararlo.

### Train

In [ ]:
# Garbage collection
import gc
for i in range(10):
  torch.cuda.empty_cache()
  gc.collect()

In [ ]:
# Train model
model = YOLO("yolov8m.pt")
history = model.train(
    data=data,
    epochs=500,
    val=True,
    imgsz=640,
    batch=-1,
    patience=100,
    time = time,
    multi_scale=True,
    weight_decay=0.0015, # Superior al anterior
    dropout=0.05,
    momentum=0.99, # Superior al anterior
)

Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.05, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.99, mosaic=1.0, multi_scale=True, name=train5, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plo

train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/train/labels.cache... 2570 images, 141 backgrounds, 0 corrupt: 100%|██████████| 2570/2570 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
AutoBatch: Computing optimal batch size for imgsz=640 at 60.0% CUDA memory utilization.
AutoBatch: CUDA:0 (Tesla T4) 14.74G total, 0.36G reserved, 0.32G allocated, 14.06G free
      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output


    25856899       79.07         1.531         51.82         41.41        (1, 3, 640, 640)                    list
    25856899       158.1         2.066         35.81          73.2        (2, 3, 640, 640)                    list
    25856899       316.3         2.930         43.61         76.15        (4, 3, 640, 640)                    list
    25856899       632.5         4.486          82.1         139.6        (8, 3, 640, 640)                    list
    25856899        1265         7.529         155.3         270.4       (16, 3, 640, 640)                    list
AutoBatch: Using batch-size 17 for CUDA:0 8.65G/14.74G (59%) ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1504.6±380.9 MB/s, size: 74.4 KB)


train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/train/labels.cache... 2570 images, 141 backgrounds, 0 corrupt: 100%|██████████| 2570/2570 [00:00<?, ?it/s]

...

      17/17      13.2G       2.04      1.355      1.459         32        480: 100%|██████████| 152/152 [01:39<00:00,  1.53it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:01<00:00,  2.29it/s]

                   all        108       3467      0.549      0.522      0.524      0.194



17 epochs completed in 0.501 hours.
Optimizer stripped from runs/detect/train5/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/train5/weights/best.pt, 52.0MB

Validating runs/detect/train5/weights/best.pt...
Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:05<00:00,  1.47s/it]


                   all        108       3467      0.549      0.522      0.525      0.194
Speed: 0.3ms preprocess, 11.0ms inference, 0.0ms loss, 4.0ms postprocess per image
Results saved to runs/detect/train5


In [ ]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e4acba27750>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.pt',
          data='/content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/data.yaml',
          epochs=500,
          time=0.5,
          patience=100,
          batch=17,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device=None,
          workers=8,
          project=None,
          name='train5',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=None,
          multi_scale=True,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.05,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
        

In [ ]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train5


### Validation

In [ ]:
# Load currently trained YOLO model
model = YOLO(f"/content/{history.save_dir}/weights/best.pt")

In [ ]:
# Validate the model
results = model.val(data=data,
          batch=64,
          conf=0.224, # best values found with optuna
          iou=0.57, # best values found with optuna
          verbose=True,
          save_json=True)

Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2049.8±404.7 MB/s, size: 92.2 KB)


val: Scanning /content/YOLO/3.5m.v4i.yolov8.640px-2steps.aug2/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.30s/it]


                   all        108       3467      0.644      0.503      0.564      0.228
Speed: 4.7ms preprocess, 23.8ms inference, 0.0ms loss, 2.2ms postprocess per image
Saving runs/detect/val5/predictions.json...
Results saved to runs/detect/val5


In [ ]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val5


In [ ]:
save_json(results)

✅ JSON file stored in: runs/detect/val5


In [ ]:
matrix = gimme_metrics(results)

Total objects detected: 4274.0
Confusion matrix:
['44.53%', '18.88%']
['36.59%', '0.00%']


### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/saveI/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/saveI/


### Metrics

In [ ]:
matrix

[[1903.0, 807.0], [1564.0, 0.0]]

In [ ]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 4274.0

Confusion matrix:
[ 44.53% , 18.88% ]
[ 36.59% , 0.00% ]

Metrics:
- Accuracy: 0.445
- Precision: 0.702
- Recall: 0.549
- F1 Score: 0.616
- F½ Score: 0.665
- G-mean: 0.621


Comparación con EXP C:
- Accuracy: Constante (0.424 ~ 0.429)
- Precision: Empeora (0.633 < 0.660) **-4.09%**
- Recall: Constante (0.562 ~ 0.550)
- F1 Score: Constante (0.595 ~ 0.600)
- F½ Score: Empeora (0.617 < 0.634)
- G-mean: Constante (0.597 ~ 0.603)

Podemos apreciar el impacto negativo de haber removido el dropout, observamos que EXP D tuvo un empeoramiento notable en Precision y F½ Score. Sin embargo, mostró una ligera mejora en Recall (+2.18%).

Comparación con Mix 3 (61):
- Accuracy: Constante (0.424 vs 0.433)
- Precision: Constante (0.633 vs 0.636
- Recall: Empeora (0.562 vs 0.576) -2.43%
- F1 Score: Constante (0.595 vs 0.605)
- F½ Score: Constante (0.617 vs 0.623)
- G-mean: Constante (0.597 vs 0.605)

**Conclusión:**  Podemos concluir que agregar un dropout pequeño mejora el entrenamiento.

# Comparación final

| Notebook        | **Accuracy** | **Precision** | **Recall** | **F1 Score** | **F½ Score** | **G-mean** | referencia |
|-----------------|--------------|---------------|------------|--------------|--------------|------------|------------|
| Exp A    | 0.438        | 0.635         | 0.584      | 0.609        | 0.625        | 0.609      |            |
| Exp B    | **0.448** | 0.634         | **0.604**      | **0.619** | 0.628 | **0.619** |            |
| Exp C    | 0.429        | **0.660**         | 0.550      | 0.600        | **0.634** | 0.603      |            |
| Exp D    | 0.424        | 0.633         | 0.562      | 0.595        | 0.617        | 0.597      |            |

# Reference

## mAP 0.553

Metrics:
- Accuracy: 0.457
- Precision: 0.680
- Recall: 0.582
- F1 Score: 0.627
- F½ Score: 0.658
- G-mean: 0.629

| Predicted Positive | Predicted Negative |
|--------------------|--------------------|
| 45.71%             | 21.47%             |
| 32.82%             | 0.00%              |